# Data Profiling: Green - NYC TLC

Este notebook realiza el perfil exploratorio de los datos de los taxis verdes (Green Cab) del proyecto NYC TLC.
Los taxis verdes (Boro Taxis) operan principalmente en los barrios de Nueva York fuera de Manhattan,
como el Bronx, Brooklyn, Queens y Staten Island, asi como en el norte de Manhattan.
El objetivo es comprender la estructura, distribucion y calidad basica de los datos en crudo.


In [1]:
import os
os.makedirs('images', exist_ok=True)
import sys
sys.path.insert(0, '/home/jovyan/work')
import os
import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
import seaborn as sns
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.types import *
from src.spark_utils import get_spark, read_parquet
from src.paths import RAW

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

VEHICLE = 'green'
RAW_PATH = str(RAW[VEHICLE])
print(f'Raw path: {RAW_PATH}')


Raw path: /home/jovyan/work/data/raw/green


## 1. Inicializacion de Spark


In [2]:
spark = get_spark(app_name=f'TLC_Profiling_{VEHICLE.upper()}')

# Leer todos los archivos parquet del directorio raw
df = read_parquet(spark, RAW_PATH)

print(f'Total registros: {df.count():,}')
print(f'Total columnas: {len(df.columns)}')
df.printSchema()


2026-07-18 18:52:01 | INFO     | tlc.spark_utils | [SPARK] Session ready | version=3.4.1 master=local[*]
2026-07-18 18:52:05 | INFO     | tlc.spark_utils | [SPARK] Read Parquet ← /home/jovyan/work/data/raw/green (Robust Mode)
Total registros: 2,038,653
Total columnas: 21
root
 |-- VendorID: long (nullable = true)
 |-- lpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- lpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)

## 2. Perfil General: Estadisticas Descriptivas


In [3]:
# Estadisticas descriptivas generales
stats = df.describe().toPandas()
print('Estadisticas descriptivas:')
try:
    display(stats)
except NameError:
    print(stats.to_string())

# Conteo de nulos por columna
total_registros = df.count()
null_counts = df.select(
    [F.sum(F.col(c).isNull().cast('int')).alias(c) for c in df.columns]
).toPandas().T
null_counts.columns = ['nulos']
null_counts['porcentaje'] = (null_counts['nulos'] / total_registros * 100).round(2)

print('\nValores nulos por columna:')
nulos_presentes = null_counts[null_counts['nulos'] > 0].sort_values('porcentaje', ascending=False)
try:
    display(nulos_presentes)
except NameError:
    print(nulos_presentes.to_string())


Estadisticas descriptivas:


,summary,VendorID,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
0,count,2038653,1908832,1908832,2038653,2038653,1908832,2038653,2038653,2038653,2038653,2038653,2038653,0,2038653,2038653,1908832,1908658,1908832,587551
1,mean,1.9309833502807983,None,1.2201429984409313,97.25044085481933,140.97078414031225,1.3003931199812242,18.373618148846308,18.268721219354294,0.8948359971020081,0.5731623037368302,2.5055368618397575,0.24574557317993792,None,0.9720676839070198,24.380507173118513,1.3083440554223735,1.0443170017886914,0.8031927377579587,0.0695173695560045
2,stddev,0.5744285674943463,None,1.7316816713949978,58.085318617421706,76.71618309519707,0.9489325051779621,1072.114129416289,17.94668831628645,1.3718418124370766,0.3688031833472843,3.4629098155989646,1.369764298206056,None,0.16716745393382293,20.01022425732454,0.49209662827011263,0.2057985114890373,1.2501084563747844,0.2175641605822277
3,min,1,N,1.0,1,1,0.0,0.0,-500.0,-7.5,-0.5,-100.0,-6.94,None,-1.0,-501.0,1.0,1.0,-2.75,-0.75
4,max,6,Y,99.0,265,265,9.0,278990.28,4003.0,12.7,61.5,460.0,108.0,None,1.0,4004.5,5.0,2.0,2.75,0.75



Valores nulos por columna:


,nulos,porcentaje
ehail_fee,2038653,100.00
cbd_congestion_fee,1451102,71.18
trip_type,129995,6.38
store_and_fwd_flag,129821,6.37
RatecodeID,129821,6.37
passenger_count,129821,6.37
payment_type,129821,6.37
congestion_surcharge,129821,6.37


## 3. Distribucion de Variables Numericas


In [4]:
# â”€â”€ OPTIMIZADO: Histogramas via Spark Bucketing (sin descargar filas crudas) â”€â”€
# En lugar de .sample(5%).toPandas() (45M filas â†’ OOM), calculamos las
# frecuencias directamente en el motor de Spark y solo descargamos los
# conteos resumidos (~40 filas) a Pandas para graficar.

from pyspark.ml.feature import Bucketizer

# Definicion de columnas, rangos y etiquetas
col_cfg = [
    ('trip_distance',   0.0, 60.0,  40, 'Distancia del viaje (millas)'),
    ('fare_amount',     0.0, 200.0, 40, 'Tarifa base (USD)'),
    ('total_amount',    0.0, 200.0, 40, 'Monto total (USD)'),
    ('tip_amount',      0.0, 60.0,  40, 'Propina (USD)'),
    ('passenger_count', 1.0, 7.0,   6,  'Cantidad de pasajeros'),
]

COLOR = 'seagreen'
fig = plt.figure(figsize=(16, 10))
gs = GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)
posiciones = [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1)]

for idx_col, (col, vmin, vmax, nbins, titulo) in enumerate(col_cfg):
    # Filtrar rango valido en Spark
    df_filt = df.filter(F.col(col).isNotNull() & (F.col(col) >= vmin) & (F.col(col) < vmax))

    # Crear splits para el Bucketizer
    import numpy as _np
    splits = list(_np.linspace(vmin, vmax, nbins + 1).tolist())
    splits[-1] = float('inf')  # capturar el extremo superior

    bkt = Bucketizer(splits=splits, inputCol=col, outputCol='_bucket')
    df_bkt = bkt.transform(df_filt.select(col))

    # Contar en Spark â†’ solo ~nbins filas se descargan a Pandas
    hist_pd = (
        df_bkt.groupBy('_bucket')
              .agg(F.count('*').alias('freq'))
              .orderBy('_bucket')
              .toPandas()
    )

    # Calcular centros de cada bin para el eje X
    step = (vmax - vmin) / nbins
    hist_pd['centro'] = vmin + (hist_pd['_bucket'] + 0.5) * step
    media_val = float(df_filt.select(F.avg(col)).first()[0] or 0)

    fila, columna = posiciones[idx_col]
    ax = fig.add_subplot(gs[fila, columna])
    ax.bar(hist_pd['centro'], hist_pd['freq'], width=step * 0.85,
           color=COLOR, edgecolor='white', alpha=0.85)
    ax.set_title(titulo, fontsize=11, fontweight='bold')
    ax.set_xlabel('Valor', fontsize=9)
    ax.set_ylabel('Frecuencia', fontsize=9)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.axvline(media_val, color='red', linestyle='--', linewidth=1.2,
               label=f'Media: {media_val:.2f}')
    ax.legend(fontsize=8)

ax_vacio = fig.add_subplot(gs[1, 2])
ax_vacio.set_visible(False)

fig.suptitle(f'Distribucion de Variables Numericas - {VEHICLE.title()} Taxi',
             fontsize=14, fontweight='bold', y=1.01)
plt.savefig(f'images/dist_numericas_{VEHICLE}.png', bbox_inches='tight', dpi=120)
plt.show()
print(f'Grafico guardado: dist_numericas_{VEHICLE}.png')


Grafico guardado: dist_numericas_green.png


## 4. Distribucion Temporal: Viajes por Ano y Mes


In [5]:
# Green taxi usa lpep_pickup_datetime (no tpep_pickup_datetime)
df_temporal = df.withColumn('anio', F.year('lpep_pickup_datetime')) \
                .withColumn('mes', F.month('lpep_pickup_datetime')) \
                .withColumn('anio_mes', F.date_format('lpep_pickup_datetime', 'yyyy-MM'))

# Agrupar y contar viajes por anio-mes
conteo_temporal = df_temporal.groupBy('anio_mes') \
    .agg(F.count('*').alias('viajes')) \
    .filter(F.col('anio_mes').isNotNull()) \
    .orderBy('anio_mes') \
    .toPandas()

# Filtrar solo anos razonables
conteo_temporal = conteo_temporal[conteo_temporal['anio_mes'].str[:4].isin(
    [str(y) for y in range(2019, 2026)]
)]

fig, ax = plt.subplots(figsize=(16, 5))
sns.barplot(data=conteo_temporal, x='anio_mes', y='viajes', color='seagreen', ax=ax)
ax.set_title('Viajes por Ano y Mes - Green Taxi', fontsize=13, fontweight='bold')
ax.set_xlabel('Periodo (Ano-Mes)', fontsize=10)
ax.set_ylabel('Cantidad de Viajes', fontsize=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Rotar etiquetas del eje x
n_etiquetas = len(conteo_temporal)
paso = max(1, n_etiquetas // 24)
etiquetas = [label if i % paso == 0 else '' for i, label in enumerate(conteo_temporal['anio_mes'])]
ax.set_xticklabels(etiquetas, rotation=45, ha='right', fontsize=8)

plt.tight_layout()
plt.savefig('images/temporal_green.png', bbox_inches='tight', dpi=120)
plt.show()
print('Grafico guardado: temporal_green.png')


Grafico guardado: temporal_green.png


## 5. Top 10 Zonas de Mayor Demanda (Pickup)


In [6]:
# Top 10 zonas de recogida por cantidad de viajes
top_zonas = df.groupBy('PULocationID') \
    .agg(F.count('*').alias('viajes')) \
    .filter(F.col('PULocationID').isNotNull()) \
    .orderBy(F.desc('viajes')) \
    .limit(10) \
    .toPandas()

top_zonas['PULocationID'] = top_zonas['PULocationID'].astype(str)
top_zonas = top_zonas.sort_values('viajes', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colores = sns.color_palette('Greens_r', n_colors=len(top_zonas))
bars = ax.barh(top_zonas['PULocationID'], top_zonas['viajes'], color=colores)

# Etiquetas de valor al final de cada barra
for bar_elem in bars:
    ancho = bar_elem.get_width()
    ax.text(ancho * 1.01, bar_elem.get_y() + bar_elem.get_height() / 2,
            f'{ancho:,.0f}', va='center', fontsize=9)

ax.set_title('Top 10 Zonas de Mayor Demanda (Pickup) - Green Taxi', fontsize=13, fontweight='bold')
ax.set_xlabel('Cantidad de Viajes', fontsize=10)
ax.set_ylabel('ID de Zona (PULocationID)', fontsize=10)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.savefig('images/top_zonas_green.png', bbox_inches='tight', dpi=120)
plt.show()
print('Grafico guardado: top_zonas_green.png')


Grafico guardado: top_zonas_green.png


## 6. Distribucion de Metodos de Pago


In [7]:
# Mapeo de codigos de pago a etiquetas legibles
mapa_pago = {
    1: 'Tarjeta',
    2: 'Efectivo',
    3: 'Sin cargo',
    4: 'Disputa',
    5: 'Desconocido',
    6: 'Anulado'
}

conteo_pago = df.groupBy('payment_type') \
    .agg(F.count('*').alias('viajes')) \
    .filter(F.col('payment_type').isNotNull()) \
    .orderBy('payment_type') \
    .toPandas()

conteo_pago['etiqueta'] = conteo_pago['payment_type'].map(
    lambda x: mapa_pago.get(int(x), f'Tipo {int(x)}') if x is not None else 'Desconocido'
)

colores_pago = sns.color_palette('Set2', n_colors=len(conteo_pago))

fig, ax = plt.subplots(figsize=(8, 8))
wedges, textos, autotextos = ax.pie(
    conteo_pago['viajes'],
    labels=conteo_pago['etiqueta'],
    autopct='%1.1f%%',
    colors=colores_pago,
    startangle=140,
    pctdistance=0.82,
    wedgeprops=dict(edgecolor='white', linewidth=1.5)
)
for texto in autotextos:
    texto.set_fontsize(10)
for texto in textos:
    texto.set_fontsize(11)

ax.set_title('Distribucion de Metodos de Pago - Green Taxi', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('images/pago_green.png', bbox_inches='tight', dpi=120)
plt.show()
print('Grafico guardado: pago_green.png')


Grafico guardado: pago_green.png


## 7. Correlacion entre Variables Numericas


In [8]:
# â”€â”€ OPTIMIZADO: Correlacion via pyspark.ml.stat (1 sola lectura del disco) â”€â”€
# El bucle doble original lanzaba 1 scan completo por cada par de columnas.
# VectorAssembler + Correlation.corr calcula la matriz entera en una unica pasada.

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

cols_corr = ['trip_distance', 'fare_amount', 'tip_amount', 'total_amount',
             'tolls_amount', 'passenger_count']

# Filtrar filas con nulos en cualquiera de las columnas de correlacion
df_corr = df.select(cols_corr).dropna()

assembler = VectorAssembler(inputCols=cols_corr, outputCol='features')
df_vec = assembler.transform(df_corr).select('features')

# Una sola pasada distribuida por Spark
corr_matrix = Correlation.corr(df_vec, 'features').head()[0].toArray()

import pandas as _pd
import numpy as _np
matriz_corr = _pd.DataFrame(corr_matrix, index=cols_corr, columns=cols_corr)

etiquetas_corr = {
    'trip_distance': 'Distancia',
    'fare_amount': 'Tarifa',
    'tip_amount': 'Propina',
    'total_amount': 'Total',
    'tolls_amount': 'Peajes',
    'passenger_count': 'Pasajeros'
}
etiquetas = [etiquetas_corr[c] for c in cols_corr]

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    matriz_corr.astype(float),
    annot=True,
    fmt='.3f',
    cmap='coolwarm',
    center=0,
    vmin=-1,
    vmax=1,
    xticklabels=etiquetas,
    yticklabels=etiquetas,
    ax=ax,
    linewidths=0.5
)
ax.set_title('Correlacion entre Variables Numericas - Green Taxi',
              fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('images/correlacion_green.png', bbox_inches='tight', dpi=120)
plt.show()
print('Grafico guardado: correlacion_green.png')


Grafico guardado: correlacion_green.png


## 8. Distribucion Horaria de Viajes


In [9]:
# Green taxi usa lpep_pickup_datetime
conteo_hora = df.withColumn('hora', F.hour('lpep_pickup_datetime')) \
    .groupBy('hora') \
    .agg(F.count('*').alias('viajes')) \
    .filter(F.col('hora').isNotNull()) \
    .orderBy('hora') \
    .toPandas()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(conteo_hora['hora'], conteo_hora['viajes'],
        marker='o', linewidth=2.2, color='seagreen', markersize=6)
ax.fill_between(conteo_hora['hora'], conteo_hora['viajes'], alpha=0.15, color='seagreen')

ax.set_title('Distribucion Horaria de Viajes - Green Taxi', fontsize=13, fontweight='bold')
ax.set_xlabel('Hora del dia (0-23)', fontsize=10)
ax.set_ylabel('Cantidad de Viajes', fontsize=10)
ax.set_xticks(range(0, 24))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.grid(True, linestyle='--', alpha=0.6)

# Marcar hora pico
hora_pico = conteo_hora.loc[conteo_hora['viajes'].idxmax(), 'hora']
viajes_pico = conteo_hora['viajes'].max()
ax.annotate(f'Hora pico: {hora_pico}h',
            xy=(hora_pico, viajes_pico),
            xytext=(hora_pico + 1.5, viajes_pico * 0.95),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=10, color='red')

plt.tight_layout()
plt.savefig('images/horario_green.png', bbox_inches='tight', dpi=120)
plt.show()
print('Grafico guardado: horario_green.png')


Grafico guardado: horario_green.png


In [10]:
spark.stop()
print('Profiling completado.')


Profiling completado.
